# Hacking Forums — 03: Análisis semántico

Este notebook trabaja con el **significado** del texto, no solo con sus estadísticas.  
Usamos modelos de IA para transformar el texto en vectores numéricos (embeddings),  
descubrir temas automáticamente (BERTopic), extraer entidades nombradas (NER),  
y sintetizar perfiles de los actores más relevantes.

### ¿Por qué embeddings?

Un **embedding** es la representación numérica de un texto en un espacio vectorial de alta dimensión  
(en nuestro caso, 4096 dimensiones). Textos semánticamente similares tienen vectores cercanos  
en ese espacio, aunque usen palabras diferentes.

Esto nos permite comparar el estilo y temática de usuarios de **distintos foros**  
en el mismo espacio matemático, independientemente del idioma.

**Prerequisito**: haber ejecutado `01_ingenieria_datos.ipynb` para tener  
`posts_clean.parquet` y `users_clean.parquet`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

from src.utils import RESULTS_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_RESULTS = RESULTS_DIR / 'hacking_forums'

posts_clean = pd.read_parquet(HF_RESULTS / 'posts_clean.parquet')
users_clean = pd.read_parquet(HF_RESULTS / 'users_clean.parquet')

print(f"posts_clean: {len(posts_clean):,} posts")
print(f"users_clean: {len(users_clean):,} usuarios")
print(f"Foros: {sorted(posts_clean['forum'].unique())}")

## Sección 1 — Embeddings con qwen3-embedding

### ¿Por qué qwen3-embedding?

`qwen3-embedding` es un modelo de embeddings **multilingüe** que produce vectores de 4096 dimensiones.  
Elegimos este modelo porque:

1. **Soporta ruso e inglés nativamente** — no necesitamos modelos distintos para Exploit.in
2. **4096 dimensiones** da alta capacidad de representación (comparado con 768 de BERT básico)
3. **Corre localmente via Ollama** — no enviamos datos a APIs externas

### Estrategia de embedding: top-50 posts más largos

Un usuario puede tener miles de posts. Embeddear todos sería muy lento.  
La estrategia: **centroides de los top-50 posts más largos** por usuario.

- Tomamos los 50 posts más largos (más contexto semántico)
- Embeddemos cada uno individualmente
- Calculamos el **centroide** (promedio vectorial) de todos

El centroide es el punto medio del espacio vectorial entre todos los posts:  
representa el "perfil promedio" del usuario.

### Caché

Embeddear miles de usuarios toma tiempo. El archivo `.npz` funciona como caché:  
si ya existe, lo cargamos directamente en lugar de recalcular.

In [ ]:
import ollama
from tqdm import tqdm

EMBEDDING_MODEL = 'qwen3:embedding'
MAX_POSTS_PER_USER = 50
EMB_OUT = HF_RESULTS / 'hacking_forums_user_embeddings.npz'

if EMB_OUT.exists():
    # Carga desde caché
    data = np.load(EMB_OUT, allow_pickle=True)
    user_ids   = data['user_ids'].tolist()
    vectors    = data['vectors']
    print(f"Cargado desde caché: {len(user_ids):,} usuarios, dim={vectors.shape[1]}")
else:
    # Calcular embeddings
    # Construir corpus por usuario: top-50 posts más largos concatenados
    posts_clean['text_len'] = posts_clean['pagetext'].astype(str).str.len()

    user_ids = []
    vectors  = []

    grouped = posts_clean.groupby(['forum', 'userid'])
    for (forum, userid), grp in tqdm(grouped, desc='Embeddings por usuario'):
        top_posts = grp.nlargest(MAX_POSTS_PER_USER, 'text_len')['pagetext'].astype(str).tolist()

        # Calcular embedding de cada post
        post_vecs = []
        for text in top_posts:
            clean_text = re.sub(r'<[^>]+>', ' ', text)[:4000]  # límite de tokens
            if len(clean_text.strip()) < 20:
                continue
            try:
                resp = ollama.embed(model=EMBEDDING_MODEL, input=clean_text)
                post_vecs.append(np.array(resp['embeddings'][0], dtype=np.float32))
            except Exception:
                continue

        if post_vecs:
            centroid = np.mean(post_vecs, axis=0)
            user_ids.append(f"{forum}_{userid}")
            vectors.append(centroid)

    vectors = np.array(vectors, dtype=np.float32)
    np.savez_compressed(EMB_OUT, user_ids=user_ids, vectors=vectors)
    print(f"Embeddings calculados y guardados: {len(user_ids):,} usuarios")

## Sección 2 — Comparativa de estrategias: Spearman ρ

### ¿Por qué comparar estrategias?

Existen varias formas de generar el vector de un usuario a partir de sus posts.  
Comparamos dos:

- **Estrategia A (embed_users)**: concatenar todos los posts en un solo texto largo y embeddear
- **Estrategia C (centroides muestreados)**: embeddear individualmente los top-50 y promediar

¿Producen rankings de similitud similares entre usuarios?

### ¿Qué mide la correlación de Spearman?

La **correlación de Spearman (ρ)** mide si dos rankings están ordenados de la misma forma.  
Si la similitud coseno entre pares de usuarios es alta con la estrategia A,  
¿también lo es con la estrategia C?

- ρ cercano a 1: las dos estrategias preservan la misma estructura
- ρ cercano a 0: las estrategias capturan señales distintas

> **Nota**: esta celda requiere haber generado embeddings con ambas estrategias.  
> Si solo existe una, la comparación se omite.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

emb_c_path = sorted(HF_RESULTS.glob('hf_centroids_sampled_*.npz'))

spearman_rho = None

if emb_c_path:
    data_c     = np.load(emb_c_path[-1], allow_pickle=True)
    user_ids_c = data_c['user_ids'].tolist()
    vectors_c  = data_c['vectors']
    print(f"Estrategia C: {len(user_ids_c):,} usuarios")

    common = sorted(set(user_ids) & set(user_ids_c))
    print(f"Usuarios en ambas estrategias: {len(common):,}")

    if len(common) >= 50:
        sample = common[:min(2000, len(common))]
        idx_a = {uid: i for i, uid in enumerate(user_ids)}
        idx_c = {uid: i for i, uid in enumerate(user_ids_c)}

        vecs_a_sub = vectors[[idx_a[u] for u in sample]]
        vecs_c_sub = vectors_c[[idx_c[u] for u in sample]]

        sim_a = cosine_similarity(vecs_a_sub)
        sim_c = cosine_similarity(vecs_c_sub)

        triu_idx = np.triu_indices(len(sample), k=1)
        rho, pval = spearmanr(sim_a[triu_idx], sim_c[triu_idx])
        spearman_rho = rho

        print(f"\nSpearman ρ = {rho:.4f}  (p = {pval:.2e})")
        if abs(rho) >= 0.7:
            print("→ Correlación fuerte: ambas estrategias preservan la estructura de similitud.")
            print("  Podemos elegir la estrategia más rápida sin sacrificar calidad.")
        elif abs(rho) >= 0.4:
            print("→ Correlación moderada: las estrategias coinciden en tendencia pero difieren en magnitud.")
        else:
            print("→ Correlación débil: las estrategias capturan señales distintas — usar ambas.")
    else:
        print("Intersección insuficiente para la comparación Spearman.")
else:
    print("No se encontró archivo de estrategia C — comparación Spearman omitida.")
    print("Continuamos con los embeddings de estrategia A (embed_users).")
    user_ids_c = user_ids
    vectors_c  = vectors

## Sección 3 — UMAP + HDBSCAN: clustering cross-foro

### ¿Qué es UMAP?

**UMAP** (Uniform Manifold Approximation and Projection) es un algoritmo que reduce  
dimensiones: toma nuestros vectores de 4096 dimensiones y los proyecta a 2 dimensiones  
preservando la estructura local de vecindad.

Si dos usuarios tienen vectores similares en 4096D, aparecerán cerca en el mapa 2D.

### ¿Qué es HDBSCAN?

**HDBSCAN** es un algoritmo de clustering (agrupación) que detecta grupos densos de puntos  
sin necesitar que le digas cuántos grupos quieres. Los puntos que no pertenecen a ningún  
grupo denso se marcan como **ruido** (cluster -1).

### ¿Qué esperamos ver?

Si hay usuarios similares temáticamente en **distintos foros**, aparecerán en el mismo cluster.  
Eso sugiere que comparten intereses o rol — independientemente del foro donde están registrados.

El parámetro `min_cluster_size=10` significa que necesitamos al menos 10 usuarios  
para que un grupo sea considerado un cluster real (no ruido).

In [ ]:
import umap as umap_lib
import hdbscan as hdbscan_lib

if len(vectors_c) >= 10:
    print("Ejecutando UMAP (puede tardar unos minutos)...")
    reducer = umap_lib.UMAP(
        n_components=2,
        random_state=42,
        n_neighbors=min(15, len(vectors_c) - 1),
        metric='cosine'
    )
    coords_2d = reducer.fit_transform(vectors_c)
    print(f"UMAP completado: {coords_2d.shape}")

    print("Ejecutando HDBSCAN...")
    clusterer = hdbscan_lib.HDBSCAN(min_cluster_size=10, min_samples=5, metric='euclidean')
    hdb_labels = clusterer.fit_predict(coords_2d)

    n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    n_noise    = (hdb_labels == -1).sum()
    print(f"Clusters encontrados: {n_clusters}")
    print(f"Puntos de ruido: {n_noise:,} ({n_noise/len(hdb_labels)*100:.1f}%)")
else:
    print("Sin embeddings disponibles para UMAP.")
    coords_2d  = np.empty((0, 2))
    hdb_labels = np.array([])

## Visualización UMAP interactiva

El siguiente gráfico muestra cada usuario como un punto en el espacio UMAP.  
El color indica el cluster al que pertenece (gris = ruido).  

Pasa el ratón sobre un punto para ver de qué foro viene.  
Si ves puntos de distintos foros en el mismo cluster, esos usuarios son semánticamente similares.

In [ ]:
if len(coords_2d) > 0 and len(hdb_labels) > 0:
    forum_labels = []
    for uid in user_ids_c:
        uid_str = str(uid)
        parts = uid_str.split('_')
        forum_labels.append(parts[0][:12] if len(parts) > 1 else 'unknown')

    palette = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
               '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

    fig = go.Figure()
    for cl in sorted(set(hdb_labels)):
        mask  = hdb_labels == cl
        color = '#444' if cl == -1 else palette[cl % len(palette)]
        name  = 'Ruido' if cl == -1 else f'Cluster {cl}'
        fig.add_trace(go.Scattergl(
            x=coords_2d[mask, 0],
            y=coords_2d[mask, 1],
            mode='markers',
            marker=dict(size=3, color=color, opacity=0.6 if cl != -1 else 0.2),
            text=[forum_labels[i] for i in np.where(mask)[0]],
            hovertemplate='%{text}<extra>' + name + '</extra>',
            name=name,
        ))

    fig.update_layout(
        title=f'UMAP + HDBSCAN — {n_clusters} clusters cross-foro · {n_noise:,} ruido',
        template='plotly_dark', height=600, width=950,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()

    # Composición de cada cluster por foro
    cluster_comp = pd.DataFrame({
        'user_id': user_ids_c,
        'forum':   forum_labels,
        'cluster': hdb_labels,
    })
    mixed = (
        cluster_comp[cluster_comp['cluster'] != -1]
        .groupby('cluster')
        .agg(n_users=('user_id', 'count'), n_forums=('forum', 'nunique'),
             forums=('forum', lambda x: ', '.join(sorted(x.unique()))))
        .query('n_forums > 1')
        .sort_values('n_users', ascending=False)
    )
    print(f"\nClusters que mezclan ≥2 foros: {len(mixed)}")
    if not mixed.empty:
        print(mixed.head(10).to_string())
else:
    print("Sin coordenadas UMAP para visualizar.")

## Sección 4 — BERTopic: topic modeling

### ¿Qué es BERTopic?

**BERTopic** es un algoritmo de topic modeling (descubrimiento de temas) que combina tres pasos:
1. Embeddear cada documento con un modelo de sentence-transformers
2. Reducir dimensiones con UMAP
3. Clusterizar con HDBSCAN

Cada cluster de documentos similares forma un **topic**. BERTopic extrae automáticamente  
las palabras más representativas de cada topic usando TF-IDF local.

### Pipeline separado para Exploit.in (ruso)

Los foros en inglés usan BERTopic con configuración `language='english'`.  
Exploit.in usa un modelo multilingüe (`paraphrase-multilingual-mpnet-base-v2`)  
que fue entrenado en 50+ idiomas incluyendo ruso.

Los topics de Exploit.in aparecerán en ruso tal como están en los posts — no se traducen.

In [ ]:
from bertopic import BERTopic

TOPIC_EN_OUT = HF_RESULTS / 'topic_hf_results.parquet'

english_posts = posts_clean[posts_clean['lang_route'] == 'en'].copy()
print(f"Posts en inglés: {len(english_posts):,}")
print(f"Foros en inglés: {sorted(english_posts['forum'].unique())}")

# Muestrear hasta 15K priorizando posts más largos
english_posts['text_len'] = english_posts['pagetext'].str.len()
english_sample = english_posts.nlargest(15_000, 'text_len').reset_index(drop=True)
print(f"\nMuestra para BERTopic: {len(english_sample):,} posts")

In [ ]:
if TOPIC_EN_OUT.exists():
    topic_en_df = pd.read_parquet(TOPIC_EN_OUT)
    print(f"Topics EN cargados desde caché: {len(topic_en_df):,} registros")
    topic_model_en = None
else:
    print("Entrenando BERTopic (foros en inglés) — puede tardar varios minutos...")
    docs_en = english_sample['pagetext'].astype(str).tolist()

    # min_topic_size=15: un topic necesita ≥15 posts para ser considerado real
    # nr_topics='auto': BERTopic decide cuántos topics crear
    topic_model_en = BERTopic(
        language='english',
        calculate_probabilities=False,
        verbose=True,
        min_topic_size=15,
        nr_topics='auto',
    )
    topics_en, _ = topic_model_en.fit_transform(docs_en)

    english_sample['topic_id'] = topics_en
    topic_info_en = topic_model_en.get_topic_info()
    n_topics = len(topic_info_en) - 1  # excluye el topic -1 (outliers)
    print(f"\nTopics encontrados: {n_topics}")
    print(topic_info_en[topic_info_en['Topic'] != -1].head(20)[['Topic', 'Count', 'Name']].to_string(index=False))

    topic_en_df = english_sample[['postid', 'forum', 'userid', 'topic_id']].copy()
    topic_en_df.to_parquet(TOPIC_EN_OUT, index=False)
    print(f"\nGuardado: {TOPIC_EN_OUT}")

## BERTopic para Exploit.in (ruso)

Exploit.in requiere un modelo de embeddings multilingüe.  
Los topics resultantes aparecen en ruso tal como están en los posts originales.

In [ ]:
from sentence_transformers import SentenceTransformer

TOPIC_RU_OUT = HF_RESULTS / 'topic_hf_ru_results.parquet'

exploit_posts = posts_clean[posts_clean['lang_route'] == 'ru'].copy()
print(f"Posts de Exploit.in (ruso): {len(exploit_posts):,}")

if len(exploit_posts) == 0:
    print("Sin posts de Exploit.in — omitiendo pipeline ruso.")
elif TOPIC_RU_OUT.exists():
    topic_ru_df = pd.read_parquet(TOPIC_RU_OUT)
    print(f"Topics RU cargados desde caché: {len(topic_ru_df):,} registros")
else:
    exploit_posts['text_len'] = exploit_posts['pagetext'].str.len()
    exploit_sample = exploit_posts.nlargest(5_000, 'text_len').reset_index(drop=True)
    docs_ru = exploit_sample['pagetext'].astype(str).tolist()
    print(f"Muestra: {len(docs_ru):,} posts")

    # Modelo multilingüe que incluye ruso con buena calidad
    embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

    topic_model_ru = BERTopic(
        embedding_model=embedding_model,
        calculate_probabilities=False,
        verbose=True,
        min_topic_size=10,
        nr_topics='auto',
    )
    topics_ru, _ = topic_model_ru.fit_transform(docs_ru)

    exploit_sample['topic_id'] = topics_ru
    topic_ru_info = topic_model_ru.get_topic_info()
    print(f"\nTopics RU encontrados: {len(topic_ru_info) - 1}")
    print(topic_ru_info[topic_ru_info['Topic'] != -1].head(10)[['Topic', 'Count', 'Name']].to_string(index=False))

    topic_ru_df = exploit_sample[['postid', 'forum', 'userid', 'topic_id']].copy()
    topic_ru_df.to_parquet(TOPIC_RU_OUT, index=False)
    print(f"Guardado: {TOPIC_RU_OUT}")

## Sección 5 — NER con qwen2.5:14b (foros en inglés)

### ¿Qué es NER?

**NER** (Named Entity Recognition) es la tarea de identificar y clasificar **entidades nombradas**  
dentro de un texto: personas, organizaciones, herramientas, malware, etc.

Usamos `qwen2.5:14b` via Ollama con un prompt estructurado que le pide devolver  
únicamente un JSON con las entidades encontradas.

### Tipos de entidades que extraemos

| Tipo | Descripción | Ejemplo |
|------|-------------|---------|  
| PERSON | Nombres reales | John Smith |
| ALIAS | Handles online | h4ck3r99 |
| TOOL | Herramientas de hacking | Metasploit, Cobalt Strike |
| MALWARE | Familias de malware | LockBit, Raccoon Stealer |
| CVE | Vulnerabilidades | CVE-2021-44228 |
| ORGANIZATION | Empresas, agencias | NSA, Google |
| MARKETPLACE | Mercados darkweb | Genesis Market |

### ¿Por qué solo foros en inglés?

`qwen2.5:14b` fue entrenado principalmente en inglés y chino.  
Su rendimiento en NER ruso es significativamente inferior.  
Exploit.in queda excluido de este pipeline.

In [ ]:
import ollama

NER_OUT        = HF_RESULTS / 'ner_hf_results.parquet'
NER_CHECKPOINT = HF_RESULTS / 'ner_hf_checkpoint.json'

TOP_N_USERS      = 50
MAX_POSTS_NER    = 50
NER_MODEL        = 'qwen2.5:14b'

NER_SYSTEM = """You are a forensic NER system specialized in underground hacking forum posts.
Extract named entities and return ONLY a valid JSON array. No explanation, no markdown.

Entity types: PERSON, ORGANIZATION, ALIAS, TOOL, MALWARE, CVE, MARKETPLACE

Return format: [{"entity": "name", "type": "TYPE"}, ...]
If no entities found, return: []"""

def extract_entities(text: str) -> list:
    """Llama a qwen2.5:14b para extraer entidades de un post."""
    prompt = f"Extract entities from this hacking forum post:\n\n{text[:2000]}"
    try:
        response = ollama.chat(
            model=NER_MODEL,
            messages=[
                {'role': 'system', 'content': NER_SYSTEM},
                {'role': 'user',   'content': prompt},
            ],
            options={'temperature': 0.0},
        )
        raw   = response['message']['content'].strip()
        start = raw.find('[')
        end   = raw.rfind(']') + 1
        if start == -1 or end == 0:
            return []
        return json.loads(raw[start:end])
    except Exception:
        return []

# Prueba rápida del modelo
test_text = "They used CVE-2021-44228 (Log4Shell) with Cobalt Strike against the target."
print("Prueba del pipeline NER:")
print(json.dumps(extract_entities(test_text), indent=2))

In [ ]:
from tqdm import tqdm

if NER_OUT.exists():
    ner_df = pd.read_parquet(NER_OUT)
    print(f"NER cargado desde caché: {len(ner_df):,} entidades")
else:
    # Top 50 usuarios por posts (solo foros en inglés)
    top_users = (
        posts_clean[posts_clean['lang_route'] == 'en']
        .groupby(['forum', 'userid'])
        .size()
        .reset_index(name='n_posts')
        .sort_values('n_posts', ascending=False)
        .head(TOP_N_USERS)
    )

    # Cargar checkpoint si existe
    ner_checkpoint = {}
    if NER_CHECKPOINT.exists():
        with open(NER_CHECKPOINT) as f:
            ner_checkpoint = json.load(f)
        print(f"Checkpoint: {len(ner_checkpoint)} usuarios ya procesados")

    ner_records = []
    for _, row in tqdm(top_users.iterrows(), total=len(top_users), desc='NER'):
        forum  = row['forum']
        userid = str(row['userid'])
        key    = f"{forum}::{userid}"

        if key in ner_checkpoint:
            ner_records.extend(ner_checkpoint[key])
            continue

        user_posts = (
            posts_clean[
                (posts_clean['forum'] == forum) &
                (posts_clean['userid'].astype(str) == userid)
            ]
            .assign(text_len=lambda df: df['pagetext'].str.len())
            .nlargest(MAX_POSTS_NER, 'text_len')
        )

        user_ents = []
        for _, post_row in user_posts.iterrows():
            for ent in extract_entities(str(post_row['pagetext'])):
                if isinstance(ent, dict) and 'entity' in ent and 'type' in ent:
                    user_ents.append({
                        'forum': forum, 'userid': userid,
                        'entity': ent['entity'], 'entity_type': ent['type'],
                    })

        ner_checkpoint[key] = user_ents
        ner_records.extend(user_ents)

    with open(NER_CHECKPOINT, 'w') as f:
        json.dump(ner_checkpoint, f, ensure_ascii=False)

    ner_df = pd.DataFrame(ner_records)
    if len(ner_df) > 0:
        ner_df.to_parquet(NER_OUT, index=False)
        print(f"\nEntidades extraídas: {len(ner_df):,}")
        print(ner_df['entity_type'].value_counts().to_string())
    else:
        print("Sin entidades — ejecutar con datos reales.")

## Sección 6 — Perfilado de actores

### La sección nueva: más allá de clasificar entidades

NER y BERTopic nos dicen **qué** se menciona y **sobre qué** se habla.  
El perfilado de actores va un paso más: **sintetiza todo lo que sabemos de un usuario**  
y le pide al modelo que infiera su **rol en el ecosistema**.

Los roles típicos en el underground:
- **Vendedor de accesos**: ofrece accesos RDP, VPN, cuentas comprometidas
- **Desarrollador de herramientas**: escribe exploits, stealers, RATs
- **Comprador / revendedor**: compra activos para revenderlos
- **Broker**: intermedia entre compradores y vendedores
- **Admin / moderador**: gestiona el foro, valida transacciones

### ¿Cómo funciona?

Para cada usuario de interés, construimos un **perfil texto** con:
- Sus entidades NER más frecuentes
- Sus topics más activos (BERTopic)
- Su conteo de posts y foros donde aparece

Luego enviamos ese perfil a `qwen2.5:14b` con un prompt de síntesis  
que le pide inferir el rol del actor.

> **Nota**: Exploit.in queda excluido del perfilado NER, pero sus embeddings  
> sí participan del clustering cross-foro de la sección anterior.

In [ ]:
PROFILING_OUT        = HF_RESULTS / 'actor_profiles.parquet'
PROFILING_CHECKPOINT = HF_RESULTS / 'actor_profiles_checkpoint.json'

PROFILE_SYSTEM = """You are a cybercrime intelligence analyst. Given a summary of a user's activity
on underground hacking forums, infer their likely role in the ecosystem.

Respond in JSON only:
{
  "role": "<one of: access_seller, tool_developer, buyer_reseller, broker, admin_mod, unknown>",
  "confidence": "<high|medium|low>",
  "rationale": "<1-2 sentences explaining the inference>",
  "key_indicators": ["<indicator1>", "<indicator2>"]
}"""

def profile_actor(user_summary: str) -> dict:
    """Infiere el rol de un actor underground a partir de su resumen de actividad."""
    try:
        response = ollama.chat(
            model=NER_MODEL,
            messages=[
                {'role': 'system', 'content': PROFILE_SYSTEM},
                {'role': 'user',   'content': f"User activity summary:\n{user_summary}"},
            ],
            options={'temperature': 0.0},
        )
        raw   = response['message']['content'].strip()
        start = raw.find('{')
        end   = raw.rfind('}') + 1
        if start == -1 or end == 0:
            return {'role': 'unknown', 'confidence': 'low', 'rationale': 'Parse error', 'key_indicators': []}
        return json.loads(raw[start:end])
    except Exception as e:
        return {'role': 'unknown', 'confidence': 'low', 'rationale': str(e), 'key_indicators': []}

print("Pipeline de perfilado listo.")

In [ ]:
from tqdm import tqdm
from collections import Counter

if PROFILING_OUT.exists():
    profiles_df = pd.read_parquet(PROFILING_OUT)
    print(f"Perfiles cargados desde caché: {len(profiles_df):,}")
else:
    # Seleccionar usuarios de interés:
    # 1. Los más activos en foros en inglés (top 30)
    # 2. Los que aparecen en múltiples foros (si pivoting_candidates.parquet existe)
    top_active = (
        posts_clean[posts_clean['lang_route'] == 'en']
        .groupby(['forum', 'userid'])
        .size()
        .reset_index(name='n_posts')
        .sort_values('n_posts', ascending=False)
        .head(30)
    )

    profiling_checkpoint = {}
    if PROFILING_CHECKPOINT.exists():
        with open(PROFILING_CHECKPOINT) as f:
            profiling_checkpoint = json.load(f)

    profile_records = []

    for _, row in tqdm(top_active.iterrows(), total=len(top_active), desc='Perfilando actores'):
        forum  = row['forum']
        userid = str(row['userid'])
        key    = f"{forum}::{userid}"

        if key in profiling_checkpoint:
            profile_records.append(profiling_checkpoint[key])
            continue

        # Recuperar username
        user_row = users_clean[
            (users_clean['forum'] == forum) &
            (users_clean['userid'].astype(str) == userid)
        ]
        username = user_row['username_raw'].iloc[0] if not user_row.empty and 'username_raw' in user_row.columns else userid

        # Entidades NER del usuario (si disponibles)
        user_entities_str = ''
        if len(ner_df) > 0 and 'userid' in ner_df.columns:
            user_ents = ner_df[
                (ner_df['forum'] == forum) & (ner_df['userid'].astype(str) == userid)
            ]
            if not user_ents.empty:
                ent_counts = user_ents.groupby(['entity_type', 'entity']).size().reset_index(name='count')
                top_ents   = ent_counts.sort_values('count', ascending=False).head(15)
                user_entities_str = top_ents.to_string(index=False)

        # Posts de muestra del usuario
        sample_posts = (
            posts_clean[
                (posts_clean['forum'] == forum) &
                (posts_clean['userid'].astype(str) == userid)
            ]
            .assign(text_len=lambda df: df['pagetext'].str.len())
            .nlargest(5, 'text_len')['pagetext']
            .astype(str).tolist()
        )
        posts_excerpt = '\n---\n'.join(p[:500] for p in sample_posts)

        summary = (
            f"Username: {username}\n"
            f"Forum: {forum}\n"
            f"Total posts: {row['n_posts']}\n"
            f"\nTop entities (NER):\n{user_entities_str}\n"
            f"\nSample posts (5 longest):\n{posts_excerpt}"
        )

        profile = profile_actor(summary)
        record = {
            'forum':    forum,
            'userid':   userid,
            'username': username,
            'n_posts':  row['n_posts'],
            **profile,
        }
        profiling_checkpoint[key] = record
        profile_records.append(record)

    with open(PROFILING_CHECKPOINT, 'w') as f:
        json.dump(profiling_checkpoint, f, ensure_ascii=False)

    profiles_df = pd.DataFrame(profile_records)
    if len(profiles_df) > 0:
        profiles_df.to_parquet(PROFILING_OUT, index=False)
        print(f"\nPerfiles generados: {len(profiles_df):,}")
        print("\nDistribución de roles:")
        print(profiles_df['role'].value_counts().to_string())
    else:
        print("Sin perfiles generados — ejecutar con datos reales.")

## Visualización de perfiles de actores

Si el perfilado corrió correctamente, mostramos la distribución de roles detectados  
y los actores con mayor confianza en la asignación.

In [ ]:
if 'profiles_df' in dir() and len(profiles_df) > 0 and 'role' in profiles_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    role_counts = profiles_df['role'].value_counts()
    axes[0].barh(role_counts.index[::-1], role_counts.values[::-1], color='#4E9EE9', alpha=0.85)
    axes[0].set_title('Distribución de roles inferidos')
    axes[0].set_xlabel('Usuarios')

    if 'n_posts' in profiles_df.columns:
        top_users_plot = profiles_df.nlargest(15, 'n_posts')
        axes[1].barh(
            top_users_plot['username'].tolist()[::-1],
            top_users_plot['n_posts'].tolist()[::-1],
            color=['#E94E4E' if r == 'access_seller' else
                   '#4EE97A' if r == 'tool_developer' else '#E9A24E'
                   for r in top_users_plot['role'].tolist()[::-1]],
            alpha=0.85
        )
        axes[1].set_title('Top 15 usuarios por posts (color = rol)')
        axes[1].set_xlabel('Posts')

    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'semantico_actor_profiles.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nPerfiles detallados (top 10 por posts):")
    cols = ['username', 'forum', 'n_posts', 'role', 'confidence', 'rationale']
    cols_avail = [c for c in cols if c in profiles_df.columns]
    print(profiles_df.nlargest(10, 'n_posts')[cols_avail].to_string(index=False))
else:
    print("Perfiles no disponibles — ejecutar la celda de perfilado primero.")

## Sección 7 — Similitud coseno cross-foro

### La pregunta clave

Tenemos candidatos de pivoting del notebook 02 (handles que coinciden entre foros).  
Ahora preguntamos: **¿los usuarios que comparten handle también comparten embedding?**

Si la respuesta es sí, tenemos **convergencia de señales** — la atribución es sólida.  
Si la respuesta es no, el handle compartido podría ser coincidencia.

### Similitud coseno

La **similitud coseno** mide el ángulo entre dos vectores en el espacio de embeddings.  
- Valor 1.0: los vectores apuntan en la misma dirección → textos semánticamente idénticos
- Valor 0.0: vectores perpendiculares → sin relación semántica
- Valor -1.0: vectores opuestos → significados contrarios

En la práctica para texto, valores por encima de 0.7 indican similitud alta.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

candidates_path = HF_RESULTS / 'pivoting_candidates.parquet'

if not candidates_path.exists():
    print("pivoting_candidates.parquet no encontrado.")
    print("Ejecutá primero el notebook 02_analisis_estructural.")
elif len(vectors_c) == 0:
    print("Sin embeddings disponibles para calcular similitud.")
else:
    candidates_df = pd.read_parquet(candidates_path)
    uid_to_idx = {uid: i for i, uid in enumerate(user_ids_c)}

    # Mapear handle+forum → userid_global
    hf_to_uid = {}
    if 'username' in users_clean.columns and 'userid' in users_clean.columns:
        for _, u_row in users_clean.iterrows():
            key = (u_row['username'], u_row['forum'])
            hf_to_uid[key] = f"{u_row['forum']}_{u_row['userid']}"

    enriched = []
    for _, pair in candidates_df.iterrows():
        uid_a = hf_to_uid.get((pair['handle_a'], pair['forum_a']))
        uid_b = hf_to_uid.get((pair['handle_b'], pair['forum_b']))
        emb_sim = None
        if uid_a and uid_b and uid_a in uid_to_idx and uid_b in uid_to_idx:
            va = vectors_c[uid_to_idx[uid_a]].reshape(1, -1)
            vb = vectors_c[uid_to_idx[uid_b]].reshape(1, -1)
            emb_sim = float(cosine_similarity(va, vb)[0, 0])
        enriched.append({**pair.to_dict(), 'embedding_sim': emb_sim})

    enriched_df = (
        pd.DataFrame(enriched)
        .dropna(subset=['embedding_sim'])
        .sort_values('embedding_sim', ascending=False)
    )

    enriched_out = HF_RESULTS / 'pivoting_candidates_enriched.parquet'
    enriched_df.to_parquet(enriched_out, index=False)

    print(f"Candidatos con embedding_sim calculado: {len(enriched_df):,}")
    print(f"\nTop 20 por similitud de embedding:")
    print(
        enriched_df[['handle_a', 'forum_a', 'handle_b', 'forum_b', 'handle_sim', 'embedding_sim']]
        .head(20)
        .to_string(index=False)
    )
    print(f"\nGuardado en: {enriched_out}")

## Resumen del análisis semántico

En este notebook aplicamos análisis con IA a los datos del corpus:

| Análisis | Modelo usado | Output |
|----------|-------------|--------|
| Embeddings de usuarios | qwen3-embedding (4096D) | `hacking_forums_user_embeddings.npz` |
| Comparativa A vs C | Spearman ρ | Score de correlación |
| Clustering cross-foro | UMAP + HDBSCAN | Clusters con mezcla de foros |
| Topic modeling (inglés) | BERTopic | `topic_hf_results.parquet` |
| Topic modeling (ruso) | BERTopic multilingüe | `topic_hf_ru_results.parquet` |
| NER (solo inglés) | qwen2.5:14b | `ner_hf_results.parquet` |
| Perfilado de actores | qwen2.5:14b | `actor_profiles.parquet` |
| Similitud cross-foro | Similitud coseno | `pivoting_candidates_enriched.parquet` |

**Siguiente paso**: notebook `04_sintesis_informe.ipynb` — convergencia de señales y conclusiones.